# TensorFlow Neural Networks Implementation - Lab 1
## Tasks 5-10: Neural Network Training and Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/AIIIT-NULP/blob/main/l1/tensorflow_models.ipynb)

This notebook implements TensorFlow neural networks for the Titanic survival prediction task, covering:
- **Task 5**: Data splitting (train/validation/test)
- **Task 6**: Basic and advanced neural network architectures
- **Task 7**: Activation function comparison (ReLU, LeakyReLU, GELU)
- **Task 8**: Model training with callbacks
- **Task 9**: Comprehensive evaluation metrics
- **Task 10**: Training curves and visualization analysis

**Runtime recommendation**: Use GPU runtime for faster training
`Runtime > Change runtime type > Hardware accelerator: GPU`

In [ ]:
# TensorFlow Implementation - Tasks 5-10
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)
if len(tf.config.list_physical_devices('GPU')) > 0:
    print("GPU devices:", tf.config.list_physical_devices('GPU'))

# Load the transformed dataset
try:
    # Try to load from current directory first
    df_final = pd.read_csv('transformed_df.csv')
    print(f"✅ Dataset loaded from local file: {df_final.shape}")
except FileNotFoundError:
    if IN_COLAB:
        print("❌ transformed_df.csv not found!")
        print("📋 Please upload the file using one of these methods:")
        print("   1. Use the file browser (📁) on the left to upload transformed_df.csv")
        print("   2. Upload to Google Drive and modify the path below:")
        print("      df_final = pd.read_csv('/content/drive/MyDrive/path/to/transformed_df.csv')")
        print("   3. Or run the dataset_preparation.ipynb notebook first")
        
        # For demo purposes, create a sample dataset notification
        print("\n🔄 Creating sample dataset for demonstration...")
        # You could recreate a minimal dataset here if needed
        df_final = None
    else:
        print("❌ transformed_df.csv not found in local directory!")
        print("Please ensure the file exists or run dataset_preparation.ipynb first")
        df_final = None

if df_final is not None:
    print("Dataset columns:", list(df_final.columns))
    print("Dataset shape:", df_final.shape)

In [1]:
# TensorFlow Implementation - Tasks 5-10
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Load the transformed dataset from the previous notebook
df_final = pd.read_csv('transformed_df.csv')
print(f"Dataset loaded: {df_final.shape}")
print("Dataset columns:", list(df_final.columns))

TensorFlow version: 2.20.0
GPU Available: []
Dataset loaded: (891, 14)
Dataset columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'IsAlone', 'AgeCategory', 'FareCategory']


In [ ]:
# TASK 5: Train/Validation/Test Split
print("="*60)
print("TASK 5: TRAIN/VALIDATION/TEST SPLIT")
print("="*60)

# Prepare features and target
X = df_final.drop('Survived', axis=1).values
y = df_final['Survived'].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution: {np.bincount(y)} (0: died, 1: survived)")

# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: 60% train, 20% validation (from the 80%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp  # 0.25 * 0.8 = 0.2
)

print(f"\nSplit results:")
print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Check class distribution in each split
train_dist = np.bincount(y_train)
val_dist = np.bincount(y_val)
test_dist = np.bincount(y_test)

print(f"\nClass distribution:")
print(f"Train: {train_dist} ({train_dist[1]/len(y_train)*100:.1f}% survived)")
print(f"Val:   {val_dist} ({val_dist[1]/len(y_val)*100:.1f}% survived)")
print(f"Test:  {test_dist} ({test_dist[1]/len(y_test)*100:.1f}% survived)")

# Convert to TensorFlow tensors
X_train_tf = tf.constant(X_train, dtype=tf.float32)
X_val_tf = tf.constant(X_val, dtype=tf.float32)
X_test_tf = tf.constant(X_test, dtype=tf.float32)
y_train_tf = tf.constant(y_train, dtype=tf.float32)
y_val_tf = tf.constant(y_val, dtype=tf.float32)
y_test_tf = tf.constant(y_test, dtype=tf.float32)

print(f"\nTensorFlow tensors created:")
print(f"X_train_tf: {X_train_tf.shape}, dtype: {X_train_tf.dtype}")
print(f"y_train_tf: {y_train_tf.shape}, dtype: {y_train_tf.dtype}")

print("✅ Task 5 completed: Train/Validation/Test split ready")

In [ ]:
# TASK 6A: Basic Feedforward Neural Network
print("="*60)
print("TASK 6A: BASIC FEEDFORWARD NEURAL NETWORK")
print("="*60)

def create_basic_model(input_dim, activation='relu'):
    """
    Create a basic feedforward neural network with minimum 2 hidden layers
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        
        # First hidden layer
        tf.keras.layers.Dense(128, activation=activation, name='hidden1'),
        
        # Second hidden layer
        tf.keras.layers.Dense(64, activation=activation, name='hidden2'),
        
        # Third hidden layer (additional layer for complexity)
        tf.keras.layers.Dense(32, activation=activation, name='hidden3'),
        
        # Output layer (binary classification)
        tf.keras.layers.Dense(1, activation='sigmoid', name='output')
    ])
    
    return model

# Create basic model with ReLU activation
input_dim = X_train.shape[1]
basic_model = create_basic_model(input_dim, activation='relu')

# Display model architecture
print(f"Input dimensions: {input_dim}")
print("\nBasic Model Architecture:")
basic_model.summary()

# Compile the model
basic_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

print("✅ Basic model created and compiled")

In [ ]:
# TASK 6B: Advanced Neural Network with Regularization
print("="*60)
print("TASK 6B: ADVANCED NETWORK WITH DROPOUT/BATCHNORM/L2")
print("="*60)

def create_advanced_model(input_dim, activation='relu', dropout_rate=0.3, l2_reg=0.01):
    """
    Create an advanced neural network with:
    - Dropout layers for regularization
    - Batch Normalization for stable training
    - L2 regularization on weights
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        
        # First hidden layer with L2 regularization
        tf.keras.layers.Dense(
            128, 
            activation=activation, 
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
            name='advanced_hidden1'
        ),
        tf.keras.layers.BatchNormalization(name='bn1'),
        tf.keras.layers.Dropout(dropout_rate, name='dropout1'),
        
        # Second hidden layer
        tf.keras.layers.Dense(
            64, 
            activation=activation,
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
            name='advanced_hidden2'
        ),
        tf.keras.layers.BatchNormalization(name='bn2'),
        tf.keras.layers.Dropout(dropout_rate, name='dropout2'),
        
        # Third hidden layer
        tf.keras.layers.Dense(
            32, 
            activation=activation,
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
            name='advanced_hidden3'
        ),
        tf.keras.layers.BatchNormalization(name='bn3'),
        tf.keras.layers.Dropout(dropout_rate * 0.5, name='dropout3'),  # Reduced dropout closer to output
        
        # Output layer
        tf.keras.layers.Dense(1, activation='sigmoid', name='advanced_output')
    ])
    
    return model

# Create advanced model with regularization
advanced_model = create_advanced_model(
    input_dim, 
    activation='relu',
    dropout_rate=0.3,
    l2_reg=0.01
)

# Display model architecture
print("Advanced Model Architecture:")
advanced_model.summary()

# Compile the advanced model
advanced_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

print("\nRegularization techniques used:")
print("  • Dropout: 0.3 rate (0.15 near output)")
print("  • Batch Normalization: After each hidden layer")
print("  • L2 Regularization: 0.01 on all dense layers")
print("  • Adam optimizer with 0.001 learning rate")

print("✅ Advanced model created and compiled")

In [ ]:
# TASK 7: Compare Activation Functions (ReLU, LeakyReLU, GELU)
print("="*60)
print("TASK 7: ACTIVATION FUNCTIONS COMPARISON")
print("="*60)

# Define activation functions to test
activation_functions = {
    'relu': 'relu',
    'leaky_relu': tf.keras.layers.LeakyReLU(alpha=0.2),
    'gelu': 'gelu'
}

# Create models with different activation functions
models = {}
for name, activation in activation_functions.items():
    print(f"\n--- Creating models with {name.upper()} activation ---")
    
    # Basic model with this activation
    basic_model_act = create_basic_model(input_dim, activation=activation)
    basic_model_act.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    
    # Advanced model with this activation
    advanced_model_act = create_advanced_model(
        input_dim, 
        activation=activation,
        dropout_rate=0.3,
        l2_reg=0.01
    )
    advanced_model_act.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    
    models[f'basic_{name}'] = basic_model_act
    models[f'advanced_{name}'] = advanced_model_act
    
    print(f"  ✓ Basic {name} model created")
    print(f"  ✓ Advanced {name} model created")

print(f"\nTotal models created: {len(models)}")
print("Model variants:")
for model_name in models.keys():
    print(f"  • {model_name}")

# Display activation function properties
print("\n--- Activation Function Properties ---")
print("ReLU: f(x) = max(0, x)")
print("  • Simple, fast computation")
print("  • Can cause 'dying ReLU' problem")
print("  • Most commonly used")

print("\nLeakyReLU: f(x) = max(αx, x) where α=0.2")
print("  • Prevents dying ReLU problem")
print("  • Small gradient for negative values")
print("  • Good for avoiding saturation")

print("\nGELU: f(x) = x * Φ(x) (Gaussian Error Linear Unit)")
print("  • Smooth, differentiable everywhere")
print("  • Used in modern transformers")
print("  • Better gradient flow")

print("✅ All activation function models created")

In [ ]:
# TASK 8: Train Models for 10-20 Epochs
print("="*60)
print("TASK 8: TRAINING ALL MODELS")
print("="*60)

# Training configuration
EPOCHS = 15
BATCH_SIZE = 32
VERBOSE = 1

# Store training histories
training_histories = {}
trained_models = {}

# Early stopping callback to prevent overfitting
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=0
)

# Reduce learning rate callback
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=0
)

print(f"Training configuration:")
print(f"  • Epochs: {EPOCHS}")
print(f"  • Batch size: {BATCH_SIZE}")
print(f"  • Early stopping: patience=5")
print(f"  • Learning rate reduction: factor=0.5, patience=3")

# Train all models
for model_name, model in models.items():
    print(f"\n{'='*40}")
    print(f"Training: {model_name.upper()}")
    print(f"{'='*40}")
    
    # Train the model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stopping, reduce_lr],
        verbose=VERBOSE
    )
    
    # Store results
    training_histories[model_name] = history
    trained_models[model_name] = model
    
    # Display final metrics
    final_epoch = len(history.history['loss'])
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    
    print(f"\n📊 Final Results for {model_name}:")
    print(f"  Training stopped at epoch: {final_epoch}")
    print(f"  Train accuracy: {final_train_acc:.4f}")
    print(f"  Validation accuracy: {final_val_acc:.4f}")
    print(f"  Train loss: {final_train_loss:.4f}")
    print(f"  Validation loss: {final_val_loss:.4f}")
    print(f"  Overfitting gap: {abs(final_train_acc - final_val_acc):.4f}")

print(f"\n🎉 All {len(models)} models trained successfully!")
print("✅ Task 8 completed: Model training finished")

In [ ]:
# TASK 9: Evaluation Metrics
print("="*60)
print("TASK 9: MODEL EVALUATION METRICS")
print("="*60)

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score

def evaluate_model(model, X_test, y_test, model_name):
    """
    Comprehensive evaluation of a trained model
    """
    # Make predictions
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = (y_pred_proba > 0.5).astype(int).flatten()
    y_pred_proba = y_pred_proba.flatten()
    
    # Calculate metrics
    test_loss, test_accuracy, test_precision, test_recall = model.evaluate(X_test, y_test, verbose=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    
    # Create results dictionary
    results = {
        'model_name': model_name,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'roc_auc': roc_auc,
        'f1_score': f1,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    return results

# Evaluate all trained models
evaluation_results = {}

print("Evaluating all models on test set...")
print(f"Test set size: {len(X_test)} samples")

for model_name, model in trained_models.items():
    print(f"\n--- Evaluating {model_name.upper()} ---")
    
    results = evaluate_model(model, X_test, y_test, model_name)
    evaluation_results[model_name] = results
    
    print(f"Test Accuracy: {results['test_accuracy']:.4f}")
    print(f"Test Precision: {results['test_precision']:.4f}")
    print(f"Test Recall: {results['test_recall']:.4f}")
    print(f"F1-Score: {results['f1_score']:.4f}")
    print(f"ROC-AUC: {results['roc_auc']:.4f}")

# Create comprehensive results comparison
print(f"\n{'='*80}")
print("COMPREHENSIVE MODEL COMPARISON")
print(f"{'='*80}")

# Create comparison DataFrame
comparison_data = []
for model_name, results in evaluation_results.items():
    comparison_data.append({
        'Model': model_name,
        'Architecture': 'Advanced' if 'advanced' in model_name else 'Basic',
        'Activation': model_name.split('_')[-1].upper(),
        'Test_Accuracy': results['test_accuracy'],
        'Test_Precision': results['test_precision'],
        'Test_Recall': results['test_recall'],
        'F1_Score': results['f1_score'],
        'ROC_AUC': results['roc_auc'],
        'Test_Loss': results['test_loss']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test_Accuracy', ascending=False)

print("\nTop 3 Best Performing Models (by Test Accuracy):")
print(comparison_df.head(3)[['Model', 'Test_Accuracy', 'F1_Score', 'ROC_AUC']].to_string(index=False))

print("\nArchitecture Comparison:")
arch_comparison = comparison_df.groupby('Architecture')[['Test_Accuracy', 'F1_Score', 'ROC_AUC']].mean()
print(arch_comparison)

print("\nActivation Function Comparison:")
activation_comparison = comparison_df.groupby('Activation')[['Test_Accuracy', 'F1_Score', 'ROC_AUC']].mean()
print(activation_comparison)

# Detailed classification report for best model
best_model_name = comparison_df.iloc[0]['Model']
best_results = evaluation_results[best_model_name]

print(f"\n{'='*50}")
print(f"DETAILED REPORT - BEST MODEL: {best_model_name.upper()}")
print(f"{'='*50}")

print("\nClassification Report:")
print(classification_report(y_test, best_results['y_pred'], 
                          target_names=['Died', 'Survived']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, best_results['y_pred'])
print(f"""
True Negatives (Correctly predicted died): {cm[0,0]}
False Positives (Incorrectly predicted survived): {cm[0,1]}
False Negatives (Incorrectly predicted died): {cm[1,0]}
True Positives (Correctly predicted survived): {cm[1,1]}
""")

print("✅ Task 9 completed: Comprehensive model evaluation finished")

In [ ]:
# TASK 10: Plot Training Curves and Loss/Metrics Graphs
print("="*60)
print("TASK 10: TRAINING CURVES AND VISUALIZATION")
print("="*60)

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid')
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# 1. TRAINING CURVES FOR ALL MODELS
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Training Curves Comparison - All Models', fontsize=16, fontweight='bold')

# Loss curves
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')

# Accuracy curves  
axes[0, 1].set_title('Training & Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')

# Precision curves
axes[0, 2].set_title('Training & Validation Precision')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Precision')

# Recall curves
axes[1, 0].set_title('Training & Validation Recall')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Recall')

# Plot curves for each model
for i, (model_name, history) in enumerate(training_histories.items()):
    color = colors[i % len(colors)]
    
    # Loss
    axes[0, 0].plot(history.history['loss'], color=color, linestyle='-', alpha=0.7, label=f'{model_name} (train)')
    axes[0, 0].plot(history.history['val_loss'], color=color, linestyle='--', alpha=0.7, label=f'{model_name} (val)')
    
    # Accuracy
    axes[0, 1].plot(history.history['accuracy'], color=color, linestyle='-', alpha=0.7, label=f'{model_name} (train)')
    axes[0, 1].plot(history.history['val_accuracy'], color=color, linestyle='--', alpha=0.7, label=f'{model_name} (val)')
    
    # Precision
    axes[0, 2].plot(history.history['precision'], color=color, linestyle='-', alpha=0.7, label=f'{model_name} (train)')
    axes[0, 2].plot(history.history['val_precision'], color=color, linestyle='--', alpha=0.7, label=f'{model_name} (val)')
    
    # Recall
    axes[1, 0].plot(history.history['recall'], color=color, linestyle='-', alpha=0.7, label=f'{model_name} (train)')
    axes[1, 0].plot(history.history['val_recall'], color=color, linestyle='--', alpha=0.7, label=f'{model_name} (val)')

# Add legends (only to first subplot to avoid clutter)
axes[0, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 2].grid(True, alpha=0.3)
axes[1, 0].grid(True, alpha=0.3)

# 2. ROC CURVES COMPARISON
axes[1, 1].set_title('ROC Curves Comparison')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')

for i, (model_name, results) in enumerate(evaluation_results.items()):
    fpr, tpr, _ = roc_curve(y_test, results['y_pred_proba'])
    auc_score = results['roc_auc']
    color = colors[i % len(colors)]
    axes[1, 1].plot(fpr, tpr, color=color, linewidth=2, 
                    label=f'{model_name} (AUC = {auc_score:.3f})')

axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(True, alpha=0.3)

# 3. PERFORMANCE METRICS COMPARISON
axes[1, 2].set_title('Test Performance Metrics')

metrics = ['Test_Accuracy', 'F1_Score', 'ROC_AUC']
x = np.arange(len(evaluation_results))
width = 0.25

for i, metric in enumerate(metrics):
    values = [evaluation_results[model]['test_accuracy'] if metric == 'Test_Accuracy'
              else evaluation_results[model]['f1_score'] if metric == 'F1_Score'
              else evaluation_results[model]['roc_auc'] for model in evaluation_results.keys()]
    
    axes[1, 2].bar(x + i*width, values, width, label=metric, alpha=0.8)

axes[1, 2].set_xlabel('Models')
axes[1, 2].set_ylabel('Score')
axes[1, 2].set_xticks(x + width)
axes[1, 2].set_xticklabels([name[:12] + '...' if len(name) > 12 else name for name in evaluation_results.keys()], 
                           rotation=45, ha='right', fontsize=8)
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Training curves visualization completed")

# 4. ARCHITECTURE & ACTIVATION COMPARISON
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Architecture and Activation Function Analysis', fontsize=16, fontweight='bold')

# Architecture comparison
arch_data = comparison_df.groupby('Architecture')[['Test_Accuracy', 'F1_Score', 'ROC_AUC']].mean()
arch_data.plot(kind='bar', ax=axes[0], rot=0, alpha=0.8)
axes[0].set_title('Performance by Architecture')
axes[0].set_ylabel('Score')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Activation function comparison
activation_data = comparison_df.groupby('Activation')[['Test_Accuracy', 'F1_Score', 'ROC_AUC']].mean()
activation_data.plot(kind='bar', ax=axes[1], rot=0, alpha=0.8)
axes[1].set_title('Performance by Activation Function')
axes[1].set_ylabel('Score')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Best model confusion matrix heatmap
best_model_name = comparison_df.iloc[0]['Model']
best_results = evaluation_results[best_model_name]
cm = confusion_matrix(y_test, best_results['y_pred'])

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
axes[2].set_title(f'Confusion Matrix\\nBest Model: {best_model_name}')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print("📈 Architecture and activation analysis completed")

# 5. SUMMARY STATISTICS
print(f"\n{'='*70}")
print("FINAL SUMMARY - TENSORFLOW IMPLEMENTATION")
print(f"{'='*70}")

print(f"\n🏆 BEST PERFORMING MODEL:")
best_model = comparison_df.iloc[0]
print(f"   Model: {best_model['Model']}")
print(f"   Architecture: {best_model['Architecture']}")
print(f"   Activation: {best_model['Activation']}")
print(f"   Test Accuracy: {best_model['Test_Accuracy']:.4f}")
print(f"   F1-Score: {best_model['F1_Score']:.4f}")
print(f"   ROC-AUC: {best_model['ROC_AUC']:.4f}")

print(f"\n📊 KEY FINDINGS:")
print(f"   • Best architecture: {arch_data.idxmax()['Test_Accuracy']}")
print(f"   • Best activation: {activation_data.idxmax()['Test_Accuracy']}")
print(f"   • Average improvement (Advanced vs Basic): {(arch_data.loc['Advanced', 'Test_Accuracy'] - arch_data.loc['Basic', 'Test_Accuracy']):.4f}")

print(f"\n🔧 REGULARIZATION EFFECTIVENESS:")
basic_avg = comparison_df[comparison_df['Architecture'] == 'Basic']['Test_Accuracy'].mean()
advanced_avg = comparison_df[comparison_df['Architecture'] == 'Advanced']['Test_Accuracy'].mean()
print(f"   • Basic models average: {basic_avg:.4f}")
print(f"   • Advanced models average: {advanced_avg:.4f}")
print(f"   • Improvement: {((advanced_avg - basic_avg) / basic_avg * 100):.2f}%")

print("\n✅ Task 10 completed: All visualizations and analysis finished")
print("🎉 ALL TENSORFLOW TASKS (5-10) COMPLETED SUCCESSFULLY!")